# PyTorch 数据序列化与存储

> 本笔记本是 [TFrecord格式.ipynb](./TFrecord格式.ipynb) 的 **PyTorch 等价版本**，
> 原版使用 TensorFlow TFRecord 进行数据序列化，本版使用 PyTorch 生态的序列化方案实现相同功能。

本教程系统讲解 PyTorch 生态中的数据序列化与存储方案，包括 torch.save/load、HDF5、WebDataset 等技术。

## 学习目标

1. 掌握 torch.save / torch.load 的张量序列化方法
2. 学会使用 HDF5 (h5py) 存储大规模数据集
3. 了解 WebDataset 格式用于大规模分布式训练
4. 对比 TFRecord 与 PyTorch 生态的序列化方案

## 1. 环境配置

In [ ]:
import shutil
import tempfile
import time
from pathlib import Path

import h5py
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset, TensorDataset

# 设置随机种子确保结果可复现 / Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# 检测并选择计算设备 / Detect and select compute device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 版本信息 / Version info
print(f"PyTorch 版本: {torch.__version__}")
print(f"h5py 版本: {h5py.__version__}")
print(f"NumPy 版本: {np.__version__}")
print(f"计算设备: {device}")

# 创建临时工作目录 / Create temporary working directory
WORK_DIR = Path(tempfile.mkdtemp(prefix="pytorch_serialization_"))
print(f"工作目录: {WORK_DIR}")

## 2. torch.save / torch.load 基础

### 2.1 保存和加载张量

`torch.save` 使用 Python 的 `pickle` 模块将对象序列化为磁盘文件，
默认格式为 PyTorch 自定义的 ZIP 格式（自 PyTorch 1.6 起）。

| 方法 | 说明 |
|------|------|
| `torch.save(obj, path)` | 将对象保存到文件 |
| `torch.load(path)` | 从文件加载对象 |

In [ ]:
# 保存单个张量 / Save a single tensor
tensor_single = torch.randn(3, 4)
single_path = WORK_DIR / "single_tensor.pt"

torch.save(tensor_single, single_path)
print(f"原始张量:\n{tensor_single}")
print(f"\n文件大小: {single_path.stat().st_size} bytes")

# 加载张量 / Load tensor
loaded_single = torch.load(single_path, weights_only=True)
print(f"\n加载后张量:\n{loaded_single}")
print(f"\n数据一致性: {torch.equal(tensor_single, loaded_single)}")

In [ ]:
# 保存张量字典 / Save a dictionary of tensors
tensor_dict = {
    "features": torch.randn(100, 10),
    "labels": torch.randint(0, 5, (100,)),
    "metadata": {
        "num_classes": 5,
        "feature_dim": 10,
    }
}

dict_path = WORK_DIR / "tensor_dict.pt"
torch.save(tensor_dict, dict_path)

# 加载字典 / Load dictionary
loaded_dict = torch.load(dict_path, weights_only=True)

print(f"features 形状: {loaded_dict['features'].shape}")
print(f"labels 形状: {loaded_dict['labels'].shape}")
print(f"metadata: {loaded_dict['metadata']}")
print(f"\n数据一致性: {torch.equal(tensor_dict['features'], loaded_dict['features'])}")

In [ ]:
# 保存张量列表 / Save a list of tensors
tensor_list = [torch.randn(5) for _ in range(3)]
list_path = WORK_DIR / "tensor_list.pt"

torch.save(tensor_list, list_path)
loaded_list = torch.load(list_path, weights_only=True)

print("张量列表保存与加载:")
for i, (orig, loaded) in enumerate(zip(tensor_list, loaded_list)):
    print(f"  张量 {i}: 一致性 = {torch.equal(orig, loaded)}")

### 2.2 保存和加载模型状态

PyTorch 推荐保存模型的 `state_dict`（参数字典），而非整个模型对象。
这种方式更安全、更灵活，且不依赖源代码结构。

In [ ]:
import torch.nn as nn


# 定义简单模型 / Define a simple model
class SimpleClassifier(nn.Module):
    """
    简单分类模型 / Simple classification model
    """
    def __init__(self, input_dim=10, hidden_dim=32, num_classes=5):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        return self.fc2(x)


model = SimpleClassifier()
print("模型结构:")
print(model)
print(f"\n参数数量: {sum(p.numel() for p in model.parameters())}")

In [ ]:
# 方式1：保存 state_dict（推荐）/ Save state_dict (recommended)
state_dict_path = WORK_DIR / "model_state_dict.pt"
torch.save(model.state_dict(), state_dict_path)

# 加载 state_dict / Load state_dict
new_model = SimpleClassifier()  # 需要先创建同结构模型 / Must create model with same structure first
new_model.load_state_dict(torch.load(state_dict_path, weights_only=True))

print(f"state_dict 文件大小: {state_dict_path.stat().st_size} bytes")

# 验证参数一致性 / Verify parameter consistency
for (name1, p1), (name2, p2) in zip(model.named_parameters(), new_model.named_parameters()):
    assert name1 == name2 and torch.equal(p1, p2), f"参数 {name1} 不一致"
print("state_dict 加载验证: 全部参数一致")

In [ ]:
# 方式2：保存整个模型（不推荐）/ Save entire model (not recommended)
entire_model_path = WORK_DIR / "model_entire.pt"
torch.save(model, entire_model_path)

print(f"state_dict 文件大小: {state_dict_path.stat().st_size} bytes")
print(f"整个模型文件大小: {entire_model_path.stat().st_size} bytes")
print("\n注意: 保存整个模型使用 pickle，加载时需要原始类定义，")
print("且不同环境间可能不兼容。推荐使用 state_dict 方式。")

### 2.3 保存预处理后的数据集

在实际项目中，数据预处理可能非常耗时（如特征提取、数据清洗等）。
将预处理后的数据保存为 `.pt` 文件，下次训练时可以直接加载，
避免重复预处理，大幅提高效率。

In [ ]:
# 模拟耗时预处理 / Simulate time-consuming preprocessing
np.random.seed(RANDOM_SEED)

# 原始数据 / Raw data
raw_features = np.random.randn(1000, 20).astype(np.float32)
raw_labels = np.random.randint(0, 5, size=1000)

print("模拟数据预处理...")
start_time = time.time()

# 模拟预处理: 标准化 + 特征工程 / Simulate preprocessing: standardization + feature engineering
mean = raw_features.mean(axis=0)
std = raw_features.std(axis=0) + 1e-8
normalized = (raw_features - mean) / std

# 特征交叉 / Feature crossing
cross_features = (normalized[:, :10] * normalized[:, 10:]).astype(np.float32)
processed_features = np.concatenate([normalized, cross_features], axis=1)

preprocess_time = time.time() - start_time
print(f"预处理耗时: {preprocess_time*1000:.2f} ms")
print(f"处理后特征维度: {processed_features.shape[1]}")

# 保存预处理后的数据 / Save preprocessed data
preprocessed_path = WORK_DIR / "preprocessed_dataset.pt"
torch.save({
    "features": torch.from_numpy(processed_features),
    "labels": torch.from_numpy(raw_labels),
    "preprocessing_params": {
        "mean": mean.tolist(),
        "std": std.tolist(),
    }
}, preprocessed_path)

print(f"\n预处理数据已保存: {preprocessed_path}")
print(f"文件大小: {preprocessed_path.stat().st_size / 1024:.1f} KB")

In [ ]:
# 加载预处理数据并创建 DataLoader / Load preprocessed data and create DataLoader
loaded_data = torch.load(preprocessed_path, weights_only=True)

features = loaded_data["features"]
labels = loaded_data["labels"]
params = loaded_data["preprocessing_params"]

print(f"加载特征: {features.shape}")
print(f"加载标签: {labels.shape}")
print(f"预处理参数均值范围: [{params['mean'][0]:.4f}, ...]")

# 直接创建 DataLoader / Create DataLoader directly
dataset = TensorDataset(features, labels)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# 验证一个批次 / Verify one batch
for batch_features, batch_labels in loader:
    print(f"\n批次特征: {batch_features.shape}")
    print(f"批次标签: {batch_labels.shape}")
    break

In [ ]:
# 完整工作流: 预处理 → 保存 → 加载 → 训练 / Full workflow: preprocess → save → load → train
def preprocess_save_load_train_demo():
    """
    演示完整的预处理-保存-加载-训练工作流
    Demonstrate complete preprocess-save-load-train workflow.
    """
    # Step 1: 加载已预处理数据 / Load preprocessed data
    data = torch.load(preprocessed_path, weights_only=True)
    features, labels = data["features"], data["labels"]

    # Step 2: 划分训练/验证集 / Split train/val sets
    n = len(features)
    indices = torch.randperm(n)
    train_idx, val_idx = indices[:800], indices[800:]

    train_ds = TensorDataset(features[train_idx], labels[train_idx])
    val_ds = TensorDataset(features[val_idx], labels[val_idx])

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=64)

    # Step 3: 创建模型并训练 / Create model and train
    clf = nn.Sequential(
        nn.Linear(features.shape[1], 64),
        nn.ReLU(),
        nn.Linear(64, 5)
    )
    optimizer = torch.optim.Adam(clf.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(5):
        clf.train()
        total_loss = 0
        for X, y in train_loader:
            optimizer.zero_grad()
            loss = criterion(clf(X), y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        if (epoch + 1) % 2 == 0:
            print(f"  Epoch {epoch+1}/5, Loss: {total_loss/len(train_loader):.4f}")

    # Step 4: 验证 / Evaluate
    clf.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X, y in val_loader:
            pred = clf(X).argmax(dim=1)
            correct += (pred == y).sum().item()
            total += len(y)
    print(f"  验证准确率: {correct/total:.4f}")

preprocess_save_load_train_demo()

### 2.4 pickle 协议与安全性

`torch.save` 底层使用 Python 的 `pickle` 进行序列化，
这意味着加载 `.pt` 文件时会执行任意 Python 代码，存在安全隐患。

| 参数 | 说明 |
|------|------|
| `weights_only=True` | 仅加载张量数据，不执行任意代码（推荐） |
| `weights_only=False` | 允许加载任意 Python 对象（默认，不安全） |

**最佳实践**:
- 加载不受信任来源的 `.pt` 文件时，务必使用 `weights_only=True`
- 只保存张量、字典等纯数据对象，避免保存自定义类实例
- PyTorch 2.0+ 推荐使用 `torch.export` 导出模型

In [ ]:
# 安全加载演示 / Safe loading demonstration

# weights_only=True: 只能加载张量、字典等基础类型 / Only load basic types like tensors, dicts
safe_data = torch.load(dict_path, weights_only=True)
print(f"weights_only=True 加载成功: {type(safe_data)}")

# weights_only=True 的限制 / Limitations of weights_only=True
print("\nweights_only=True 的限制:")
print("  - 不支持加载自定义类实例")
print("  - 不支持加载 lambda 函数")
print("  - 仅支持: 张量、字典、列表、元组、基本类型")

# 保存 state_dict 本身就是安全做法 / Saving state_dict itself is a safe practice
print("\n推荐做法:")
print("  1. 保存 state_dict 而非整个模型")
print("  2. 加载时使用 weights_only=True")
print("  3. 不要加载不受信任来源的 .pt 文件")

## 3. HDF5 (h5py) 大规模数据存储

### 3.1 HDF5 基础概念

HDF5 (Hierarchical Data Format 5) 是一种用于存储和组织大规模科学数据的文件格式。
与 TFRecord 和 torch.save 相比，HDF5 具有独特优势：

| 特性 | HDF5 | torch.save | TFRecord |
|------|------|------------|----------|
| 随机访问 | 支持 | 不支持（需全部加载） | 不支持 |
| 分块读取 | 支持（chunked） | 不支持 | 不支持 |
| 压缩 | gzip, lzf, szip | 无内置 | GZIP, ZLIB |
| 层级结构 | Groups + Datasets | 扁平字典 | 扁平记录 |
| 元数据 | Attributes | 需额外存储 | Protocol Buffers |
| 部分读取 | 支持切片 | 不支持 | 不支持 |

**核心概念**:
- **Groups**: 类似文件系统的目录，用于层级组织
- **Datasets**: 类似文件系统中的文件，存储实际的多维数组
- **Attributes**: 附加在 Group 或 Dataset 上的元数据
- **Chunked Storage**: 将数据分块存储，支持部分 I/O

In [ ]:
# HDF5 基本结构探索 / Explore HDF5 basic structure
h5_demo_path = WORK_DIR / "demo.h5"

with h5py.File(h5_demo_path, 'w') as f:
    # 创建组（类似目录）/ Create groups (like directories)
    train_group = f.create_group('train')
    val_group = f.create_group('val')

    # 在组中创建数据集 / Create datasets in groups
    train_group.create_dataset('features', data=np.random.randn(100, 10).astype(np.float32))
    train_group.create_dataset('labels', data=np.random.randint(0, 5, 100))

    val_group.create_dataset('features', data=np.random.randn(20, 10).astype(np.float32))
    val_group.create_dataset('labels', data=np.random.randint(0, 5, 20))

    # 添加属性（元数据）/ Add attributes (metadata)
    f.attrs['description'] = '演示数据集'
    f.attrs['version'] = '1.0'
    train_group.attrs['num_samples'] = 100
    train_group.attrs['num_classes'] = 5

# 读取 HDF5 文件结构 / Read HDF5 file structure
def print_h5_structure(name, obj):
    """
    打印 HDF5 文件结构 / Print HDF5 file structure.

    Parameters:
    -----------
    name : str
        对象路径 / Object path
    obj : h5py object
        HDF5 对象 / HDF5 object
    """
    if isinstance(obj, h5py.Dataset):
        print(f"  Dataset: {name} | shape={obj.shape} | dtype={obj.dtype}")
    elif isinstance(obj, h5py.Group):
        attrs = dict(obj.attrs)
        print(f"  Group:   {name} | attrs={attrs}")

with h5py.File(h5_demo_path, 'r') as f:
    print(f"HDF5 文件: {h5_demo_path}")
    print(f"全局属性: {dict(f.attrs)}")
    print("\n文件结构:")
    f.visititems(print_h5_structure)

### 3.2 创建 HDF5 数据集

HDF5 支持多种压缩算法和分块存储策略，
可以显著减少存储空间并优化读取性能。

In [ ]:
# 压缩与分块存储 / Compression and chunked storage

# 生成测试数据 / Generate test data
large_data = np.random.randn(10000, 256).astype(np.float32)  # ~10 MB

h5_compress_path = WORK_DIR / "compression_demo.h5"

with h5py.File(h5_compress_path, 'w') as f:
    # 不压缩 / No compression
    f.create_dataset('raw', data=large_data)

    # gzip 压缩（压缩比高，速度适中）/ gzip compression (high ratio, moderate speed)
    f.create_dataset('gzip', data=large_data,
                     compression='gzip', compression_opts=9)

    # lzf 压缩（速度快，压缩比适中）/ lzf compression (fast, moderate ratio)
    f.create_dataset('lzf', data=large_data,
                     compression='lzf')

    # 分块存储（支持部分读取）/ Chunked storage (supports partial reads)
    f.create_dataset('chunked', data=large_data,
                     chunks=(100, 256),  # 每块 100 行 / 100 rows per chunk
                     compression='gzip', compression_opts=4)

# 对比存储大小 / Compare storage sizes
print("HDF5 压缩效果对比:")
print("-" * 50)
raw_size = large_data.nbytes
print(f"原始数据大小: {raw_size / 1024:.1f} KB")

with h5py.File(h5_compress_path, 'r') as f:
    for name in ['raw', 'gzip', 'lzf', 'chunked']:
        ds = f[name]
        stored_size = ds.id.get_storage_size()
        ratio = (1 - stored_size / raw_size) * 100
        chunks_info = f"chunks={ds.chunks}" if ds.chunks else "contiguous"
        print(f"{name:10} | {stored_size/1024:8.1f} KB | "
              f"压缩率 {ratio:5.1f}% | {chunks_info}")

### 3.3 从 HDF5 读取数据

HDF5 最大的优势之一是支持**部分读取**（Partial I/O），
无需将整个数据集加载到内存，只读取需要的切片。

In [ ]:
# 从 HDF5 读取数据 / Read data from HDF5

with h5py.File(h5_compress_path, 'r') as f:
    # 读取整个数据集 / Read entire dataset
    all_data = f['raw'][:]
    print(f"完整读取: shape={all_data.shape}, dtype={all_data.dtype}")

    # 切片读取（部分 I/O，仅读取需要的部分）/ Slice read (partial I/O)
    slice_data = f['raw'][0:10]  # 只读前10行 / Read only first 10 rows
    print(f"切片读取 [0:10]: shape={slice_data.shape}")

    # 读取特定列 / Read specific columns
    col_data = f['raw'][:, 0:5]  # 只读前5列 / Read only first 5 columns
    print(f"列切片 [:, 0:5]: shape={col_data.shape}")

    # 按索引读取 / Read by index
    indices = [0, 5, 10, 15]
    indexed_data = f['raw'][indices]
    print(f"索引读取: shape={indexed_data.shape}")

    # 验证: HDF5 Dataset 对象不占用大量内存 / HDF5 Dataset objects don't consume much memory
    ds = f['raw']
    print(f"\nDataset 对象: {type(ds)}")
    print(f"Dataset 形状: {ds.shape}")
    print("Dataset 在磁盘上，不占用 Python 堆内存")

In [ ]:
# 分块读取的性能优势 / Performance advantage of chunked reads

with h5py.File(h5_compress_path, 'r') as f:
    # 从 chunked 数据集读取一个块 / Read one chunk from chunked dataset
    start = time.time()
    chunk = f['chunked'][0:100]  # 恰好一个 chunk / Exactly one chunk
    chunk_time = time.time() - start
    print(f"读取一个 chunk (100行): {chunk_time*1000:.2f} ms")

    # 从非分块数据集读取相同数据 / Read same data from non-chunked dataset
    start = time.time()
    raw_chunk = f['raw'][0:100]
    raw_time = time.time() - start
    print(f"读取相同数据 (raw): {raw_time*1000:.2f} ms")

    print("\n注意: 分块存储的主要优势在于:")
    print("  1. 支持部分读取，无需加载整个数据集")
    print("  2. 配合压缩，可以减少磁盘 I/O")
    print("  3. 在大数据集中按需读取，节省内存")

### 3.4 自定义 HDF5Dataset

将 HDF5 文件封装为 PyTorch `Dataset`，
可以无缝集成到 `DataLoader` 训练流水线中。

In [ ]:
class HDF5Dataset(Dataset):
    """
    从 HDF5 文件读取数据的 PyTorch Dataset / PyTorch Dataset that reads from HDF5 files.

    Parameters:
    -----------
    filepath : str or Path
        HDF5 文件路径 / Path to HDF5 file
    feature_key : str
        特征数据集的键名 / Key name for feature dataset
    label_key : str
        标签数据集的键名 / Key name for label dataset
    split : str, optional
        数据集划分 ('train'/'val'/'test')，对应 HDF5 中的 Group / Dataset split
    transform : callable, optional
        特征变换函数 / Feature transform function
    cache_size : int, optional
        缓存到内存的样本数 (0=不缓存) / Number of samples to cache in memory
    """

    def __init__(self, filepath, feature_key='features', label_key='labels',
                 split=None, transform=None, cache_size=0):
        self.filepath = str(filepath)
        self.feature_key = feature_key
        self.label_key = label_key
        self.split = split
        self.transform = transform
        self.cache_size = cache_size
        self._cache = {}

        # 读取数据集信息（不加载全部数据）/ Read dataset info without loading all data
        with h5py.File(self.filepath, 'r') as f:
            group = f[split] if split else f
            self._length = group[feature_key].shape[0]
            self._feature_shape = group[feature_key].shape[1:]
            self._label_shape = group[label_key].shape[1:]

    def __len__(self):
        return self._length

    def __getitem__(self, idx):
        # 检查缓存 / Check cache
        if idx in self._cache:
            return self._cache[idx]

        # 从 HDF5 读取 / Read from HDF5
        with h5py.File(self.filepath, 'r') as f:
            group = f[self.split] if self.split else f
            feature = group[self.feature_key][idx]
            label = group[self.label_key][idx]

        # 转换为 PyTorch 张量 / Convert to PyTorch tensors
        feature = torch.from_numpy(feature)
        if isinstance(label, np.ndarray):
            label = torch.from_numpy(label)
        else:
            label = torch.tensor(label, dtype=torch.long)

        # 应用变换 / Apply transform
        if self.transform:
            feature = self.transform(feature)

        # 缓存 / Cache
        if self.cache_size > 0 and len(self._cache) < self.cache_size:
            self._cache[idx] = (feature, label)

        return feature, label


# 使用 HDF5Dataset / Using HDF5Dataset
h5_dataset = HDF5Dataset(
    h5_demo_path,
    split='train',
    feature_key='features',
    label_key='labels'
)

print(f"数据集大小: {len(h5_dataset)}")
print(f"特征形状: {h5_dataset._feature_shape}")

# 读取单个样本 / Read a single sample
feature, label = h5_dataset[0]
print(f"\n样本 0: feature={feature.shape}, label={label}")

# 创建 DataLoader / Create DataLoader
h5_loader = DataLoader(h5_dataset, batch_size=16, shuffle=True)
for features_batch, labels_batch in h5_loader:
    print(f"\nDataLoader 批次: features={features_batch.shape}, labels={labels_batch.shape}")
    break

In [ ]:
# 创建支持 train/val/test 划分的 HDF5 文件 / Create HDF5 file with train/val/test splits
split_h5_path = WORK_DIR / "split_dataset.h5"

np.random.seed(RANDOM_SEED)
n_total = 1000
n_train = 800
n_val = 100
n_test = 100

all_features = np.random.randn(n_total, 20).astype(np.float32)
all_labels = np.random.randint(0, 5, n_total)

with h5py.File(split_h5_path, 'w') as f:
    f.attrs['description'] = '带划分的数据集'
    f.attrs['num_classes'] = 5

    for split_name, start, end in [('train', 0, n_train),
                                    ('val', n_train, n_train + n_val),
                                    ('test', n_train + n_val, n_total)]:
        grp = f.create_group(split_name)
        grp.create_dataset('features', data=all_features[start:end],
                           compression='gzip', chunks=(100, 20))
        grp.create_dataset('labels', data=all_labels[start:end],
                           compression='gzip', chunks=(100,))
        grp.attrs['num_samples'] = end - start

print(f"已创建带划分的 HDF5 文件: {split_h5_path}")
print(f"文件大小: {split_h5_path.stat().st_size / 1024:.1f} KB")

# 使用不同划分创建 DataLoader / Create DataLoaders for different splits
train_ds = HDF5Dataset(split_h5_path, split='train')
val_ds = HDF5Dataset(split_h5_path, split='val')
test_ds = HDF5Dataset(split_h5_path, split='test')

print(f"\n训练集: {len(train_ds)} 样本")
print(f"验证集: {len(val_ds)} 样本")
print(f"测试集: {len(test_ds)} 样本")

### 3.5 性能对比: HDF5 vs raw 文件

对比不同存储方式的读取速度和存储空间。

In [ ]:
# 性能对比: HDF5 vs torch.save vs numpy / Performance comparison
np.random.seed(RANDOM_SEED)
bench_data = np.random.randn(10000, 100).astype(np.float32)
bench_labels = np.random.randint(0, 10, 10000)

results = {}

# 1. torch.save / torch.load
pt_path = WORK_DIR / "bench.pt"
torch.save({
    'features': torch.from_numpy(bench_data),
    'labels': torch.from_numpy(bench_labels)
}, pt_path)

start = time.time()
loaded_pt = torch.load(pt_path, weights_only=True)
pt_load_time = time.time() - start
results['torch.save'] = {
    'size_kb': pt_path.stat().st_size / 1024,
    'load_time_ms': pt_load_time * 1000
}

# 2. HDF5 (无压缩) / HDF5 (no compression)
h5_raw_path = WORK_DIR / "bench_raw.h5"
with h5py.File(h5_raw_path, 'w') as f:
    f.create_dataset('features', data=bench_data)
    f.create_dataset('labels', data=bench_labels)

start = time.time()
with h5py.File(h5_raw_path, 'r') as f:
    features_h5 = f['features'][:]
    labels_h5 = f['labels'][:]
h5_raw_time = time.time() - start
results['HDF5 (raw)'] = {
    'size_kb': h5_raw_path.stat().st_size / 1024,
    'load_time_ms': h5_raw_time * 1000
}

# 3. HDF5 (gzip 压缩) / HDF5 (gzip compressed)
h5_gz_path = WORK_DIR / "bench_gzip.h5"
with h5py.File(h5_gz_path, 'w') as f:
    f.create_dataset('features', data=bench_data, compression='gzip')
    f.create_dataset('labels', data=bench_labels, compression='gzip')

start = time.time()
with h5py.File(h5_gz_path, 'r') as f:
    features_h5gz = f['features'][:]
    labels_h5gz = f['labels'][:]
h5_gz_time = time.time() - start
results['HDF5 (gzip)'] = {
    'size_kb': h5_gz_path.stat().st_size / 1024,
    'load_time_ms': h5_gz_time * 1000
}

# 4. NumPy .npy / .npz
npz_path = WORK_DIR / "bench.npz"
np.savez(npz_path, features=bench_data, labels=bench_labels)

start = time.time()
loaded_npz = np.load(npz_path)
npz_time = time.time() - start
results['NumPy (.npz)'] = {
    'size_kb': npz_path.stat().st_size / 1024,
    'load_time_ms': npz_time * 1000
}

# 输出对比结果 / Print comparison results
print(f"数据规模: {bench_data.shape}, {bench_data.nbytes/1024:.1f} KB")
print("\n" + "="*65)
print(f"{'格式':15} | {'文件大小 (KB)':>14} | {'加载时间 (ms)':>14}")
print("-" * 65)
raw_kb = bench_data.nbytes / 1024
for name, info in results.items():
    size_ratio = info['size_kb'] / raw_kb * 100
    print(f"{name:15} | {info['size_kb']:10.1f} ({size_ratio:4.0f}%) | {info['load_time_ms']:12.2f}")
print("="*65)

## 4. WebDataset 大规模数据

### 4.1 WebDataset 设计理念

[WebDataset](https://github.com/webdataset/webdataset) 是专为大规模分布式训练设计的数据格式，
其核心思想是使用 POSIX tar 归档文件替代大量小文件。

**为什么需要 WebDataset？**

| 问题 | 传统方式 | WebDataset 方案 |
|------|----------|----------------|
| 大量小文件 | 数百万个独立文件 | 少量 tar 归档 |
| 云存储 I/O | 每个文件一次 HTTP 请求 | 流式读取 tar 文件 |
| 随机访问开销 | 文件系统瓶颈 | 顺序读取优化 |
| 分布式训练 | 需要共享文件系统 | 直接从 S3/GS 读取 |

**核心设计**:
- 基于 POSIX tar 格式，与云存储兼容
- 顺序访问模式，优化网络传输
- 每个 tar 文件包含同一样本的不同模态数据
- 文件命名约定关联同一训练样本

In [ ]:
# WebDataset 基本概念演示 / WebDataset basic concept demonstration
# 注意: 实际使用需要安装 webdataset 库: pip install webdataset
# Note: Requires webdataset library: pip install webdataset

import io
import tarfile

from PIL import Image

# 手动创建 tar 格式数据集（模拟 WebDataset 格式）/ Manually create tar dataset
# WebDataset 的约定: 同一样本的文件共享相同前缀
# WebDataset convention: files of the same sample share the same prefix

tar_path = WORK_DIR / "webdataset_sample.tar"

with tarfile.open(tar_path, 'w') as tar:
    for i in range(10):
        # 创建图像数据 / Create image data
        img = Image.fromarray(
            np.random.randint(0, 256, (32, 32, 3), dtype=np.uint8)
        )
        img_buf = io.BytesIO()
        img.save(img_buf, format='PNG')
        img_bytes = img_buf.getvalue()

        # 添加图像文件 / Add image file
        img_info = tarfile.TarInfo(name=f"{i:05d}.png")
        img_info.size = len(img_bytes)
        tar.addfile(img_info, io.BytesIO(img_bytes))

        # 添加标签文件 / Add label file
        label_str = str(np.random.randint(0, 10)).encode('utf-8')
        label_info = tarfile.TarInfo(name=f"{i:05d}.cls")
        label_info.size = len(label_str)
        tar.addfile(label_info, io.BytesIO(label_str))

        # 添加 JSON 元数据 / Add JSON metadata
        import json
        meta = json.dumps({"id": i, "source": "demo"}).encode('utf-8')
        meta_info = tarfile.TarInfo(name=f"{i:05d}.json")
        meta_info.size = len(meta)
        tar.addfile(meta_info, io.BytesIO(meta))

print(f"WebDataset tar 文件已创建: {tar_path}")
print(f"文件大小: {tar_path.stat().st_size / 1024:.1f} KB")
print("\n文件结构 (同一前缀 = 同一样本):")
print("  00000.png  <- 图像")
print("  00000.cls  <- 标签")
print("  00000.json <- 元数据")
print("  00001.png")
print("  00001.cls")
print("  ...")

### 4.2 WebDataset 基本用法

使用 `webdataset` 库可以方便地读取 tar 格式数据集，
并直接转换为 PyTorch DataLoader。

In [ ]:
# 使用 webdataset 库读取 / Read with webdataset library
# 安装: pip install webdataset

try:
    import webdataset as wds
    HAS_WEBDATASET = True
    print(f"webdataset 版本: {wds.__version__}")
except ImportError:
    HAS_WEBDATASET = False
    print("webdataset 未安装，展示概念代码")
    print("安装命令: pip install webdataset")

if HAS_WEBDATASET:
    # 从 tar 文件创建数据集 / Create dataset from tar file
    dataset = wds.WebDataset(str(tar_path))

    # 查看原始样本 / View raw samples
    for sample in dataset:
        print(f"样本键: {list(sample.keys())}")
        print(f"  .png: {type(sample['png'])}, {len(sample['png'])} bytes")
        print(f"  .cls: {sample['cls'].decode()}")
        print(f"  .json: {sample['json'].decode()}")
        break
else:
    # 概念代码 / Conceptual code
    print("\n概念代码 (需要安装 webdataset):")
    print("""
    import webdataset as wds

    # 创建数据集
    dataset = wds.WebDataset("path/to/data-{000000..000099}.tar")

    # 定义解码管道
    dataset = (dataset
        .decode("pil")            # 自动解码图像
        .to_tuple("png;jpg", "cls")  # 提取图像和标签
    )

    # 创建 DataLoader
    loader = DataLoader(dataset, batch_size=16)
    """)

### 4.3 与 DataLoader 集成

WebDataset 可以直接与 PyTorch DataLoader 集成，
构建从云存储到 GPU 的高效数据流水线。

In [ ]:
# WebDataset + DataLoader 完整管道 / WebDataset + DataLoader complete pipeline

if HAS_WEBDATASET:
    import webdataset as wds

    # 定义转换函数 / Define transform function
    def preprocess(sample):
        """
        预处理样本 / Preprocess sample.

        Parameters:
        -----------
        sample : dict
            WebDataset 样本字典 / WebDataset sample dictionary

        Returns:
        --------
        tuple : (image_tensor, label) / (image tensor, label)
        """
        image = sample['png']  # PIL Image
        image = torch.from_numpy(np.array(image)).permute(2, 0, 1).float() / 255.0
        label = int(sample['cls'].decode())
        return image, label

    # 构建管道 / Build pipeline
    dataset = (wds.WebDataset(str(tar_path))
        .decode("pil")
        .map(preprocess)
    )

    loader = DataLoader(dataset, batch_size=4)

    for images, labels in loader:
        print(f"批次图像: {images.shape}")
        print(f"批次标签: {labels}")
        break
else:
    print("完整管道概念代码 (需要安装 webdataset):")
    print("""
    # 从 S3/GS 云存储读取 / Read from S3/GS cloud storage
    dataset = (wds.WebDataset("s3://bucket/data-{000000..000099}.tar")
        .shuffle(1000)           # 缓冲区打乱
        .decode("pil")           # 解码图像
        .map(preprocess)         # 自定义预处理
        .batched(16)             # 批量化
    )

    # 分布式训练: 每个 worker 读取不同分片 / Distributed: each worker reads different shards
    # WebDataset 自动处理分片分配
    loader = DataLoader(dataset, num_workers=4)
    """)

print("\nWebDataset 优势总结:")
print("  1. 云原生: 直接从 S3/GS/azure 流式读取")
print("  2. 分布式: 自动分片分配，无需手动管理")
print("  3. 高效: 顺序读取，避免大量小文件 I/O")
print("  4. 灵活: 支持多模态数据 (图像+文本+标签)")

## 5. 实战: MNIST 数据集序列化

### 5.1 使用 torch.save 保存

In [ ]:
# 使用 torchvision 加载 MNIST / Load MNIST with torchvision
from torchvision import datasets, transforms

# 下载并加载 MNIST / Download and load MNIST
mnist_train = datasets.MNIST(
    root=str(WORK_DIR / "mnist_raw"),
    train=True,
    download=True,
    transform=transforms.ToTensor()
)

# 取前 2000 个样本用于演示 / Use first 2000 samples for demo
demo_size = 2000
print(f"MNIST 训练集大小: {len(mnist_train)}")
print(f"演示使用样本数: {demo_size}")

# 提取数据和标签 / Extract data and labels
images_list = []
labels_list = []
for i in range(demo_size):
    img, label = mnist_train[i]
    images_list.append(img)  # (1, 28, 28)
    labels_list.append(label)

images_tensor = torch.stack(images_list)  # (N, 1, 28, 28)
labels_tensor = torch.tensor(labels_list)

print(f"\n图像张量: {images_tensor.shape}, {images_tensor.dtype}")
print(f"标签张量: {labels_tensor.shape}, {labels_tensor.dtype}")
print(f"图像值域: [{images_tensor.min():.3f}, {images_tensor.max():.3f}]")

In [ ]:
# 使用 torch.save 保存 MNIST / Save MNIST with torch.save
mnist_pt_path = WORK_DIR / "mnist_demo.pt"

torch.save({
    'images': images_tensor,
    'labels': labels_tensor,
    'metadata': {
        'num_samples': demo_size,
        'image_size': (1, 28, 28),
        'num_classes': 10,
    }
}, mnist_pt_path)

pt_size_kb = mnist_pt_path.stat().st_size / 1024
print("torch.save 格式:")
print(f"  文件大小: {pt_size_kb:.1f} KB")
print(f"  每样本: {pt_size_kb * 1024 / demo_size:.1f} bytes")

### 5.2 使用 HDF5 保存

In [ ]:
# 使用 HDF5 保存 MNIST / Save MNIST with HDF5
mnist_h5_path = WORK_DIR / "mnist_demo.h5"

# 将图像转为 numpy / Convert images to numpy
images_np = images_tensor.numpy()  # (N, 1, 28, 28)
labels_np = labels_tensor.numpy()

with h5py.File(mnist_h5_path, 'w') as f:
    f.attrs['description'] = 'MNIST demo dataset'
    f.attrs['num_classes'] = 10

    # 保存为 uint8 以节省空间（原始像素值 0-255）/ Save as uint8 to save space
    images_uint8 = (images_np * 255).astype(np.uint8)  # 转回 uint8
    labels_int = labels_np.astype(np.int64)

    f.create_dataset('images', data=images_uint8,
                     compression='gzip', compression_opts=4,
                     chunks=(100, 1, 28, 28))
    f.create_dataset('labels', data=labels_int,
                     compression='gzip',
                     chunks=(100,))

    # 添加训练/验证划分 / Add train/val split
    n_train = 1600
    train_grp = f.create_group('train')
    train_grp.create_dataset('images', data=images_uint8[:n_train],
                             compression='gzip', chunks=(100, 1, 28, 28))
    train_grp.create_dataset('labels', data=labels_int[:n_train],
                             compression='gzip', chunks=(100,))

    val_grp = f.create_group('val')
    val_grp.create_dataset('images', data=images_uint8[n_train:],
                           compression='gzip', chunks=(100, 1, 28, 28))
    val_grp.create_dataset('labels', data=labels_int[n_train:],
                           compression='gzip', chunks=(100,))

h5_size_kb = mnist_h5_path.stat().st_size / 1024
print("HDF5 格式 (gzip 压缩):")
print(f"  文件大小: {h5_size_kb:.1f} KB")
print(f"  每样本: {h5_size_kb * 1024 / demo_size:.1f} bytes")
print(f"\n压缩率: {(1 - h5_size_kb / pt_size_kb) * 100:.1f}% (相比 torch.save)")

### 5.3 加载与训练

In [ ]:
# 从 torch.save 格式加载 / Load from torch.save format
print("=" * 50)
print("从 torch.save 加载")
print("=" * 50)

mnist_data = torch.load(mnist_pt_path, weights_only=True)
images_pt = mnist_data['images']
labels_pt = mnist_data['labels']

print(f"图像: {images_pt.shape}, {images_pt.dtype}")
print(f"标签: {labels_pt.shape}, {labels_pt.dtype}")

# 创建 DataLoader / Create DataLoader
pt_dataset = TensorDataset(images_pt, labels_pt)
pt_loader = DataLoader(pt_dataset, batch_size=64, shuffle=True)

for batch_imgs, batch_lbls in pt_loader:
    print(f"批次: images={batch_imgs.shape}, labels={batch_lbls.shape}")
    break

In [ ]:
# 从 HDF5 格式加载 / Load from HDF5 format
print("=" * 50)
print("从 HDF5 加载")
print("=" * 50)

# 使用自定义 HDF5Dataset / Using custom HDF5Dataset
class MNISTHDF5Dataset(Dataset):
    """
    从 HDF5 加载 MNIST 的 Dataset / Dataset for loading MNIST from HDF5.

    Parameters:
    -----------
    filepath : str or Path
        HDF5 文件路径 / Path to HDF5 file
    split : str
        数据划分 ('train'/'val') / Data split
    """

    def __init__(self, filepath, split='train'):
        self.filepath = str(filepath)
        self.split = split
        with h5py.File(self.filepath, 'r') as f:
            self._length = f[split]['images'].shape[0]

    def __len__(self):
        return self._length

    def __getitem__(self, idx):
        with h5py.File(self.filepath, 'r') as f:
            image = f[self.split]['images'][idx]  # (1, 28, 28), uint8
            label = f[self.split]['labels'][idx]

        # 转换为 float 并归一化 / Convert to float and normalize
        image = torch.from_numpy(image).float() / 255.0
        label = torch.tensor(label, dtype=torch.long)
        return image, label


h5_train_ds = MNISTHDF5Dataset(mnist_h5_path, split='train')
h5_val_ds = MNISTHDF5Dataset(mnist_h5_path, split='val')

print(f"训练集: {len(h5_train_ds)} 样本")
print(f"验证集: {len(h5_val_ds)} 样本")

# 验证数据 / Verify data
img, lbl = h5_train_ds[0]
print(f"\n样本 0: image={img.shape}, label={lbl.item()}")
print(f"图像值域: [{img.min():.3f}, {img.max():.3f}]")

# 创建 DataLoader / Create DataLoader
h5_train_loader = DataLoader(h5_train_ds, batch_size=64, shuffle=True)
h5_val_loader = DataLoader(h5_val_ds, batch_size=64)

for batch_imgs, batch_lbls in h5_train_loader:
    print(f"\nHDF5 批次: images={batch_imgs.shape}, labels={batch_lbls.shape}")
    break

In [ ]:
# 数据完整性验证 / Data integrity verification
print("数据完整性验证:")
print("-" * 40)

# 比较 torch.save 和 HDF5 加载的数据 / Compare data loaded from torch.save and HDF5
# torch.save 保存的是 float32 (0-1)，HDF5 保存的是 uint8 (0-255) 需转换
with h5py.File(mnist_h5_path, 'r') as f:
    h5_images = f['images'][:]  # uint8

# 将 torch.save 的数据转回 uint8 比较 / Convert torch.save data back to uint8 for comparison
pt_images_uint8 = (images_pt.numpy() * 255).astype(np.uint8)

match = np.array_equal(pt_images_uint8, h5_images)
print(f"图像数据一致性: {match}")

# 验证标签 / Verify labels
with h5py.File(mnist_h5_path, 'r') as f:
    h5_labels = f['labels'][:]

labels_match = np.array_equal(labels_pt.numpy(), h5_labels)
print(f"标签数据一致性: {labels_match}")

print("\n文件大小对比:")
print(f"  torch.save: {pt_size_kb:.1f} KB")
print(f"  HDF5 (gzip): {h5_size_kb:.1f} KB")
print(f"  HDF5 节省空间: {(1 - h5_size_kb / pt_size_kb) * 100:.1f}%")

## 6. 清理资源

In [ ]:
# 清理临时文件 / Clean up temporary files
shutil.rmtree(WORK_DIR)
print(f"已清理: {WORK_DIR}")

## 小结

### 序列化方法总结

| 方法 | 适用场景 | 优势 | 劣势 |
|------|----------|------|------|
| `torch.save/load` | 模型权重、中小数据集 | 简单易用、PyTorch 原生 | 不支持部分读取、pickle 安全风险 |
| HDF5 (h5py) | 大规模科学数据 | 随机访问、压缩、分块读取 | 需要额外库、并发写入受限 |
| WebDataset | 大规模分布式训练 | 云原生、流式读取、自动分片 | 仅顺序访问、需要 tar 格式 |
| NumPy (.npz) | NumPy 数据交换 | 兼容性好 | 不支持部分读取、功能有限 |

### 选择建议

1. **模型权重**: 始终使用 `torch.save(model.state_dict(), path)`
2. **中小数据集 (<1GB)**: `torch.save` 足够，简单高效
3. **大规模数据集 (>1GB)**: HDF5，支持部分读取和压缩
4. **分布式训练 + 云存储**: WebDataset，原生支持 S3/GS
5. **安全加载**: 始终使用 `weights_only=True`

## TF vs PyTorch 对照

| 概念 | TensorFlow / Keras | PyTorch |
|------|-------------------|---------|
| 原生序列化格式 | TFRecord (二进制记录流) | torch.save (pickle + ZIP) |
| 结构化数据格式 | `tf.train.Example` (Protocol Buffers) | Python dict / `state_dict` |
| 写入工具 | `tf.io.TFRecordWriter` | `torch.save(obj, path)` |
| 读取工具 | `tf.data.TFRecordDataset` | 自定义 `HDF5Dataset` / `torch.load` |
| 压缩选项 | GZIP / ZLIB / 不压缩 | HDF5: gzip / lzf / szip |
| 分片策略 | 手动 TFRecord 分片 (`data-00000-of-00100`) | WebDataset tar 分片 (`shard-000000.tar`) |
| 序列化协议 | Protocol Buffers (跨语言、高效) | pickle (Python 专属、灵活) |
| 云存储集成 | `tf.data` + GCS 原生支持 | WebDataset + S3/GS (通过 `webdataset` 库) |
| 随机访问 | TFRecord 不支持 | HDF5 支持（分块读取） |
| 部分读取 | TFRecord 不支持（需全量解析） | HDF5 支持切片读取 |
| 安全加载 | TFRecord 天然安全（Protobuf 解析） | `torch.load(weights_only=True)` |
| 元数据 | `tf.train.Features` + Attributes | HDF5 Attributes / dict 键值 |
| 分布式读取 | `interleave` + 分片 | WebDataset 自动分片 / HDF5 独立读取 |
| 大规模替代方案 | TFRecord | WebDataset (tar) / HDF5 / LMDB |

## 练习

### 练习1: 自定义图像+标签序列化格式

创建一个自定义的序列化格式，将图像（numpy 数组）和标签配对保存：

```python
# 要求:
# 1. 使用 torch.save 保存图像和标签对
# 2. 图像存储为 uint8 以节省空间
# 3. 包含元数据（图像尺寸、类别数等）
# 4. 加载后自动转换为 float32 并归一化到 [0, 1]

def save_image_label_pairs(filepath, images, labels, metadata=None):
    """保存图像-标签对"""
    # TODO: 实现保存逻辑
    pass

def load_image_label_pairs(filepath):
    """加载图像-标签对，自动归一化"""
    # TODO: 实现加载逻辑
    pass
```

思考: 与 TFRecord 的 `tf.train.Example` 相比，这种方式的优缺点是什么？

### 练习2: 对比 HDF5 chunk 大小对读取性能的影响

实验不同的 chunk 大小如何影响 HDF5 的读取性能：

```python
# 要求:
# 1. 创建一个较大的 HDF5 数据集 (e.g., 50000 x 100 float32)
# 2. 使用不同的 chunk 大小: (100, 100), (1000, 100), (10000, 100)
# 3. 测量以下场景的读取时间:
#    a) 读取单个样本 (1 行)
#    b) 读取一个 chunk
#    c) 读取整个数据集
# 4. 对比文件大小和压缩效果

chunk_sizes = [(100, 100), (1000, 100), (10000, 100)]
for chunks in chunk_sizes:
    # TODO: 创建数据集并测量性能
    pass
```

思考: chunk 大小如何影响顺序读取 vs 随机读取的性能？

### 练习3: 实现跨多个 HDF5 文件的懒加载 Dataset

实现一个可以从多个 HDF5 文件中懒加载数据的 Dataset：

```python
class MultiFileHDF5Dataset(Dataset):
    """
    跨多个 HDF5 文件的懒加载 Dataset / Lazy-loading Dataset across multiple HDF5 files.

    Parameters:
    -----------
    filepaths : list
        HDF5 文件路径列表 / List of HDF5 file paths
    feature_key : str
        特征数据集键名 / Feature dataset key
    label_key : str
        标签数据集键名 / Label dataset key
    """

    def __init__(self, filepaths, feature_key='features', label_key='labels'):
        # TODO: 初始化，建立全局索引映射
        # 提示: 需要记录每个文件的样本数，建立 (file_idx, local_idx) 映射
        pass

    def __len__(self):
        # TODO: 返回所有文件的总样本数
        pass

    def __getitem__(self, idx):
        # TODO: 根据 idx 找到对应的文件和局部索引
        # 提示: 使用 bisect 模块进行高效索引查找
        pass

# 测试
# 创建3个 HDF5 文件，每个包含不同数量的样本
# 使用 MultiFileHDF5Dataset 加载所有数据
```

思考: 这种设计如何支持分布式训练中的数据分片？每个 worker 应该读取哪些文件？